In [9]:
from rdkit import Chem
from rdkit.Chem import AllChem,Descriptors
import numpy as np
from rdkit.Chem import rdMolDescriptors

In [10]:
smiles="CC(=O)O"
mol=Chem.MolFromSmiles(smiles)
print("SMILES:",smiles);
print("分子式:", rdMolDescriptors.CalcMolFormula(mol))
print("重原子数:",mol.GetNumHeavyAtoms())

SMILES: CC(=O)O
分子式: C2H4O2
重原子数: 4


In [11]:
def get_atom_list(mol):
    atom_list=[]
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum()>1:
            atom_list.append(atom.GetSymbol())
    return atom_list
print("原子列表：",get_atom_list(mol));

原子列表： ['C', 'C', 'O', 'O']


In [21]:
def build_adj(mol):
    # 1. 获取重原子列表
    heavy_atoms = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() > 1:
            heavy_atoms.append(atom)
    n = len(heavy_atoms)

    # 2. 创建索引映射
    idx_map = {}
    i = 0
    for atom in heavy_atoms:
        idx_map[atom.GetIdx()] = i
        i = i + 1

    # 3. 初始化邻接矩阵
    A = np.zeros((n, n, 5), dtype=np.float32)

    # 4. 遍历所有键，填充矩阵
    for bond in mol.GetBonds():
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()

        if begin_idx in idx_map and end_idx in idx_map:
            i = idx_map[begin_idx]
            j = idx_map[end_idx]

            bond_type = bond.GetBondType()
            if bond_type == Chem.rdchem.BondType.SINGLE:
                k = 1
            elif bond_type == Chem.rdchem.BondType.DOUBLE:
                k = 2
            elif bond_type == Chem.rdchem.BondType.TRIPLE:
                k = 3
            elif bond_type == Chem.rdchem.BondType.AROMATIC:
                k = 4
            else:
                k = 0

            A[i, j, k] = 1
            A[j, i, k] = 1

    # ===== 注意：下面这部分必须在 for 循环外面，顶格写 =====
    # 返回邻接矩阵和原子符号列表
    atom_symbols = []
    for atom in heavy_atoms:
        atom_symbols.append(atom.GetSymbol())
    return A, atom_symbols;

A, atom_symbols = build_adj(mol)
print("原子符号:", atom_symbols)
print("邻接矩阵形状:", A.shape)
print("邻接矩阵:")
print(np.argmax(A, axis=2))

原子符号: ['C', 'C', 'O', 'O']
邻接矩阵形状: (4, 4, 5)
邻接矩阵:
[[0 1 0 0]
 [1 0 2 1]
 [0 2 0 0]
 [0 1 0 0]]


In [16]:
def get_fingerprint(mol):
    # 1. 生成 Morgan 指纹（半径2，长度2048）
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    # 2. 转换成 NumPy 数组（0/1 向量）
    fp_array = np.array(fp)
    # 3. 返回
    return fp_array
fp = get_fingerprint(mol)
print("指纹长度:", len(fp))
print("指纹中1的个数:", sum(fp))
print("前20位:", fp[:20])

指纹长度: 2048
指纹中1的个数: 7
前20位: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
